# IV.2.2 — Analisis Karakteristik Spasial Data Skeleton BISINDO

Notebook ini menghasilkan visualisasi untuk mendokumentasikan **masalah/karakteristik spasial** yang ditemukan pada data skeleton BISINDO sebelum pre-processing.

Tiga kelompok analisis:
1. **IV.2.2.1 — Variasi Posisi dan Skala Antar Penutur** → Scatter overlay, boxplot jarak keypoint, overlay skeleton multi-penutur
2. **IV.2.2.2 — Instabilitas Koordinat Antar Frame (Micro-shift)** → Line chart fluktuasi koordinat, histogram displacement
3. **IV.2.2.3 — Noise Koordinat [0,0]** → Heatmap frekuensi noise per keypoint, bar chart noise per penutur, visualisasi frame dengan noise

---

In [ ]:
import pickle
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.ticker as mticker
from collections import defaultdict
import os

# ── Paper-style global rcParams (consistent with reference notebooks) ─────────
plt.rcParams.update({
    'figure.facecolor'   : 'white',
    'axes.facecolor'     : 'white',
    'axes.edgecolor'     : '#333333',
    'axes.linewidth'     : 0.8,
    'axes.grid'          : False,
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif'],
    'font.size'          : 10,
    'axes.titlesize'     : 10,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 9,
    'xtick.labelsize'    : 8,
    'ytick.labelsize'    : 8,
    'legend.fontsize'    : 9,
    'figure.titlesize'   : 11,
    'figure.titleweight' : 'bold',
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
})

# ── Color palette ─────────────────────────────────────────────────────────────
C_LH     = '#2166AC'   # Left hand  – deep blue
C_RH     = '#D6604D'   # Right hand – muted red
C_EDGE   = '#555555'   # Skeleton edges
C_NOISE  = '#B2182B'   # Noise highlight – red
C_VALID  = '#4393C3'   # Valid keypoint
GRAYS    = ['#1a1a1a', '#555555', '#888888', '#bbbbbb']
BAR_EC   = 'black'
BAR_LW   = 0.6

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = '../data/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Hand skeleton edges ───────────────────────────────────────────────────────
HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]
CONNECTIONS = [(u,v) for u,v in HAND_EDGES] + [(u+21,v+21) for u,v in HAND_EDGES]

IDX_LH   = np.arange(0, 21)    # Left hand
IDX_RH   = np.arange(21, 42)   # Right hand
IDX_HAND = np.arange(0, 42)    # Both hands

print('Configuration loaded.')

## 0. Load Data


In [ ]:
# ── CONFIGURATION — sesuaikan path dengan struktur direktori kamu ─────────────
PICKLE_FILE = '../data/pickle/pose_bisindo.pkl'

with open(PICKLE_FILE, 'rb') as f:
    data = pickle.load(f)

video_ids = list(data.keys())
print(f'Total videos : {len(video_ids)}')
print(f'Sample IDs   : {video_ids[:5]}')

# Ekstrak info penutur dari video ID (format: Pxx_Sxxx_Rxx)
speakers = sorted(set(v.split('_')[0] for v in video_ids))
print(f'Signers      : {speakers}')

# Keypoint layout (86 titik):
# [0:21]  = Left Hand  (21 pts)
# [21:42] = Right Hand (21 pts)
# [42:61] = Mouth      (19 pts)
# [61:86] = Pose/Body  (25 pts)
K_TOTAL = 86

---
## IV.2.2.1 — Variasi Posisi dan Skala Antar Penutur

Tiga visualisasi:
- **Fig A** — Scatter plot overlay koordinat tangan semua penutur (raw, sebelum normalisasi)
- **Fig B** — Boxplot jarak wrist → middle finger tip per penutur (variasi skala tangan)
- **Fig C** — Overlay visualisasi skeleton beberapa penutur dalam satu frame


In [ ]:
# ── Fig A: Scatter Overlay Koordinat Tangan Semua Penutur ─────────────────────
# Menunjukkan bahwa posisi tangan di ruang koordinat tidak konsisten antar penutur

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=150)

titles = ['(a) Left Hand Keypoints', '(b) Right Hand Keypoints']
idx_parts = [IDX_LH, IDX_RH]
colors_sp = plt.cm.tab10(np.linspace(0, 0.9, len(speakers)))

for ax, idx_part, title in zip(axes, idx_parts, titles):
    for sp, color in zip(speakers, colors_sp):
        sp_vids = [v for v in video_ids if v.startswith(sp)]
        xs, ys = [], []
        for vid in sp_vids:
            kp = data[vid]['keypoints']  # (T, K, 2)
            pts = kp[:, idx_part, :]     # (T, 21, 2)
            # exclude [0,0] noise
            valid = ~((pts[:,:,0] == 0) & (pts[:,:,1] == 0))
            xs.extend(pts[:,:,0][valid].tolist())
            ys.extend(pts[:,:,1][valid].tolist())
        ax.scatter(xs, ys, s=0.3, alpha=0.15, color=color, label=sp, rasterized=True)

    ax.set_title(title)
    ax.set_xlabel('X Coordinate (normalized)')
    ax.set_ylabel('Y Coordinate (normalized)')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

# Legend with bigger markers
handles = [Line2D([0],[0], marker='o', color='w',
                  markerfacecolor=plt.cm.tab10(i/len(speakers)),
                  markersize=6, label=sp)
           for i, sp in enumerate(speakers)]
fig.legend(handles=handles, title='Signer', loc='center right',
           bbox_to_anchor=(1.02, 0.5), frameon=True,
           framealpha=0.9, edgecolor='#cccccc', fancybox=False)

plt.suptitle('Hand Keypoint Position Distribution per Signer (Raw Coordinates)',
             fontsize=11, fontweight='bold')
plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_A_scatter_overlay.png'))
plt.show()
print('[SAVED] spatial_A_scatter_overlay.png')

In [ ]:
# ── Fig B: Boxplot Jarak Wrist → Middle Finger Tip per Penutur ────────────────
# Keypoint: wrist LH=0, middle tip LH=12 | wrist RH=21, middle tip RH=33
# Jarak Euclidean ini merepresentasikan 'ukuran tangan' relatif per penutur

WRIST_LH, TIP_LH = 0, 12
WRIST_RH, TIP_RH = 21, 33

sp_dist_lh = defaultdict(list)
sp_dist_rh = defaultdict(list)

for vid in video_ids:
    sp = vid.split('_')[0]
    kp = data[vid]['keypoints']  # (T, K, 2)
    for t in range(kp.shape[0]):
        # Left hand
        w, tip = kp[t, WRIST_LH], kp[t, TIP_LH]
        if not (w[0]==0 and w[1]==0) and not (tip[0]==0 and tip[1]==0):
            sp_dist_lh[sp].append(np.linalg.norm(tip - w))
        # Right hand
        w, tip = kp[t, WRIST_RH], kp[t, TIP_RH]
        if not (w[0]==0 and w[1]==0) and not (tip[0]==0 and tip[1]==0):
            sp_dist_rh[sp].append(np.linalg.norm(tip - w))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0), dpi=150)
colors_box = ['#D1E5F0', '#FDDBC7']
C_MED = '#B2182B'

for ax, sp_dist, title, fill_c in zip(
    axes,
    [sp_dist_lh, sp_dist_rh],
    ['(a) Left Hand', '(b) Right Hand'],
    colors_box
):
    sp_sorted = sorted(sp_dist.keys())
    bdata = [sp_dist[s] for s in sp_sorted]
    bp = ax.boxplot(
        bdata, labels=sp_sorted, patch_artist=True,
        medianprops=dict(color=C_MED, linewidth=1.6),
        whiskerprops=dict(linewidth=0.8),
        capprops=dict(linewidth=0.8),
        flierprops=dict(marker='o', markersize=2.5,
                        alpha=0.35, markeredgecolor='none',
                        markerfacecolor='#888888')
    )
    for patch in bp['boxes']:
        patch.set_facecolor(fill_c)
    ax.set_title(title)
    ax.set_xlabel('Signer ID')
    ax.set_ylabel('Wrist–Tip Distance (normalized)')
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.suptitle('Hand Scale Variation Across Signers\n(Wrist–to–Middle Fingertip Distance)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_B_boxplot_scale.png'))
plt.show()
print('[SAVED] spatial_B_boxplot_scale.png')

In [ ]:
# ── Fig C: Overlay Skeleton Beberapa Penutur (Satu Kalimat yang Sama) ─────────
# Menunjukkan variasi posisi & ukuran secara visual intuitif
# Pilih 1 sentence ID yang sama, ambil 1 representative frame per penutur

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
TARGET_SENTENCE = 'S001'    # Ganti dengan sentence ID yang ada di datamu
TARGET_REP      = 'R01'
FRAME_IDX       = 10        # Frame ke berapa yang diambil
# ──────────────────────────────────────────────────────────────────────────────

candidate_vids = [
    v for v in video_ids
    if TARGET_SENTENCE in v and TARGET_REP in v
]
candidate_vids = sorted(candidate_vids)
print(f'Candidate videos for {TARGET_SENTENCE}_{TARGET_REP}: {candidate_vids}')

n_sp = len(candidate_vids)
fig, axes = plt.subplots(1, n_sp, figsize=(2.8 * n_sp, 3.5), dpi=150)
if n_sp == 1:
    axes = [axes]

margin = 0.1

for ax, vid in zip(axes, candidate_vids):
    sp_label = vid.split('_')[0]
    kp = data[vid]['keypoints']
    fidx = min(FRAME_IDX, kp.shape[0] - 1)
    frame = kp[fidx].copy()  # (K, 2)
    valid = ~((frame[:,0] == 0) & (frame[:,1] == 0))

    # RAW coordinates — no centering, to show positional variation
    for u, v in CONNECTIONS:
        if valid[u] and valid[v]:
            ax.plot([frame[u,0], frame[v,0]], [-frame[u,1], -frame[v,1]],
                    color=C_EDGE, linewidth=1.0, alpha=0.6, zorder=1)

    idx_lh_v = IDX_LH[valid[IDX_LH]]
    idx_rh_v = IDX_RH[valid[IDX_RH]]
    ax.scatter(frame[idx_lh_v,0], -frame[idx_lh_v,1],
               s=25, color=C_LH, edgecolors='white', linewidths=0.3, zorder=2)
    ax.scatter(frame[idx_rh_v,0], -frame[idx_rh_v,1],
               s=25, color=C_RH, edgecolors='white', linewidths=0.3, zorder=2)

    ax.set_title(f'{sp_label}\nF{fidx}', fontsize=8.5)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.05, 0.05)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.5); spine.set_color('#aaaaaa')

legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_LH,
           markersize=7, label='Left Hand'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_RH,
           markersize=7, label='Right Hand'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.04), frameon=False)

plt.suptitle(f'Signer Skeleton Overlay — Sentence {TARGET_SENTENCE} (Raw Coordinates)',
             fontsize=11, fontweight='bold')
plt.subplots_adjust(wspace=0.06, bottom=0.15)
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_C_skeleton_overlay.png'))
plt.show()
print('[SAVED] spatial_C_skeleton_overlay.png')

---
## IV.2.2.2 — Instabilitas Koordinat Antar Frame (Micro-shift)

Dua visualisasi:
- **Fig D** — Line chart fluktuasi koordinat X/Y satu keypoint tangan sepanjang sequence
- **Fig E** — Histogram displacement antar frame per keypoint tangan


In [ ]:
# ── Fig D: Line Chart Fluktuasi Koordinat Antar Frame (Micro-shift) ──────────
# Menunjukkan bahwa koordinat keypoint tidak stabil meski gerakan harusnya mulus

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
TARGET_VIDEO   = 'P01_S001_R01'   # Ganti sesuai video yang ada di datamu
KEYPOINTS_SHOW = {
    'LH Wrist (0)'        : 0,
    'LH Index Tip (8)'    : 8,
    'RH Wrist (21)'       : 21,
    'RH Index Tip (29)'   : 29,
}
FRAME_RANGE = (20, 60)   # Tampilkan frame 20–60 agar detail micro-shift terlihat
# ──────────────────────────────────────────────────────────────────────────────

kp = data[TARGET_VIDEO]['keypoints']  # (T, K, 2)
fs, fe = FRAME_RANGE
frames_plot = np.arange(fs, min(fe, kp.shape[0]))

fig, axes = plt.subplots(2, 2, figsize=(11, 5.5), dpi=150, sharex=True)
axes = axes.flatten()

line_colors = ['#2166AC', '#4393C3', '#D6604D', '#F4A582']

for ax, (label, kp_idx), color in zip(axes, KEYPOINTS_SHOW.items(), line_colors):
    x_coords = kp[frames_plot, kp_idx, 0]
    y_coords = kp[frames_plot, kp_idx, 1]

    # Mask noise [0,0]
    noise_mask = (x_coords == 0) & (y_coords == 0)

    ax.plot(frames_plot, x_coords, color=color, linewidth=1.0,
            label='X coordinate', zorder=2)
    ax.plot(frames_plot, y_coords, color=color, linewidth=1.0,
            linestyle='--', alpha=0.7, label='Y coordinate', zorder=2)

    # Highlight noise frames
    for fidx in frames_plot[noise_mask]:
        ax.axvline(fidx, color=C_NOISE, linewidth=0.7, alpha=0.5, zorder=1)

    ax.set_title(label, fontsize=9)
    ax.set_ylabel('Coordinate Value', fontsize=8)
    ax.grid(axis='y', alpha=0.25, linewidth=0.4)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

for ax in axes[2:]:
    ax.set_xlabel('Frame Index')

# Shared legend
legend_handles = [
    Line2D([0],[0], color='#555555', linewidth=1.0, label='X coordinate'),
    Line2D([0],[0], color='#555555', linewidth=1.0, linestyle='--', label='Y coordinate'),
    Line2D([0],[0], color=C_NOISE,   linewidth=1.0, label='Noise [0,0] frame'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False)

plt.suptitle(f'Keypoint Coordinate Fluctuation Across Frames — {TARGET_VIDEO}\n'
             f'(Frames {fs}–{fe})',
             fontsize=11, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_D_microshift_linechart.png'))
plt.show()
print('[SAVED] spatial_D_microshift_linechart.png')

In [ ]:
# ── Fig E: Histogram Inter-frame Displacement per Keypoint Tangan ─────────────
# Distribusi selisih koordinat frame-ke-frame untuk setiap keypoint
# Menunjukkan bahwa micro-shift tersebar merata (bukan outlier sesekali)

# Hitung displacement antar frame untuk semua video
displacements_lh = defaultdict(list)  # kp_idx -> list of displacements
displacements_rh = defaultdict(list)

for vid in video_ids:
    kp = data[vid]['keypoints']  # (T, K, 2)
    for kp_idx in range(42):
        coords = kp[:, kp_idx, :]  # (T, 2)
        for t in range(1, coords.shape[0]):
            prev, curr = coords[t-1], coords[t]
            # Exclude noise frames
            if (prev[0]==0 and prev[1]==0) or (curr[0]==0 and curr[1]==0):
                continue
            d = np.linalg.norm(curr - prev)
            if kp_idx < 21:
                displacements_lh[kp_idx].append(d)
            else:
                displacements_rh[kp_idx - 21].append(d)

# Aggregate: mean displacement per keypoint
mean_disp_lh = [np.mean(displacements_lh[i]) if displacements_lh[i] else 0 for i in range(21)]
mean_disp_rh = [np.mean(displacements_rh[i]) if displacements_rh[i] else 0 for i in range(21)]

# Distribusi overall displacement (semua keypoint tangan digabung)
all_disp_lh = [d for kp_dists in displacements_lh.values() for d in kp_dists]
all_disp_rh = [d for kp_dists in displacements_rh.values() for d in kp_dists]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), dpi=150)

# Left: histogram distribusi displacement LH
axes[0].hist(all_disp_lh, bins=60, color='#2166AC', alpha=0.8,
             edgecolor='white', linewidth=0.3)
axes[0].set_title('(a) Left Hand — Displacement Distribution')
axes[0].set_xlabel('Inter-frame Displacement (normalized)')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, np.percentile(all_disp_lh, 98))  # Clip outlier

# Middle: histogram RH
axes[1].hist(all_disp_rh, bins=60, color='#D6604D', alpha=0.8,
             edgecolor='white', linewidth=0.3)
axes[1].set_title('(b) Right Hand — Displacement Distribution')
axes[1].set_xlabel('Inter-frame Displacement (normalized)')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, np.percentile(all_disp_rh, 98))

# Right: mean displacement per keypoint index
x = np.arange(21)
w = 0.35
axes[2].bar(x - w/2, mean_disp_lh, width=w, color='#2166AC',
            edgecolor='white', linewidth=0.3, label='Left Hand')
axes[2].bar(x + w/2, mean_disp_rh, width=w, color='#D6604D',
            edgecolor='white', linewidth=0.3, label='Right Hand')
axes[2].set_title('(c) Mean Displacement per Keypoint Index')
axes[2].set_xlabel('Keypoint Index (within hand)')
axes[2].set_ylabel('Mean Displacement')
axes[2].legend(frameon=False)

for ax in axes:
    ax.grid(axis='y', alpha=0.25, linewidth=0.4)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.suptitle('Inter-frame Displacement Distribution — Hand Keypoints',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_E_displacement_histogram.png'))
plt.show()
print('[SAVED] spatial_E_displacement_histogram.png')

---
## IV.2.2.3 — Noise Koordinat [0,0]

Tiga visualisasi:
- **Fig F** — Heatmap frekuensi noise [0,0] per keypoint (semua 86 titik)
- **Fig G** — Bar chart persentase frame dengan noise per penutur
- **Fig H** — Visualisasi frame dengan keypoint [0,0] (ditandai merah)


In [ ]:
# ── Fig F: Heatmap Frekuensi Noise [0,0] per Keypoint ────────────────────────
# Menunjukkan bagian tubuh mana yang paling sering tidak terdeteksi MediaPipe

BODY_PARTS = {
    'Left Hand\n(0–20)'   : (0, 21),
    'Right Hand\n(21–41)' : (21, 42),
    'Mouth\n(42–60)'      : (42, 61),
    'Pose\n(61–85)'       : (61, 86),
}

# Hitung frekuensi noise per keypoint
noise_count  = np.zeros(K_TOTAL)
total_frames = np.zeros(K_TOTAL)

for vid in video_ids:
    kp = data[vid]['keypoints']  # (T, K, 2)
    T  = kp.shape[0]
    is_noise = (kp[:,:,0] == 0) & (kp[:,:,1] == 0)  # (T, K)
    noise_count  += is_noise.sum(axis=0)
    total_frames += T

noise_rate = noise_count / total_frames  # proporsi frame dengan noise per keypoint

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 3.5), dpi=150)

x = np.arange(K_TOTAL)
colors_part = ['#2166AC', '#D6604D', '#4DAF4A', '#984EA3']
part_colors = np.empty(K_TOTAL, dtype=object)
for (label, (start, end)), c in zip(BODY_PARTS.items(), colors_part):
    part_colors[start:end] = c

bars = ax.bar(x, noise_rate * 100, width=0.85,
              color=part_colors, edgecolor='none')

# Partition boundaries
for (label, (start, end)), c in zip(BODY_PARTS.items(), colors_part):
    mid = (start + end) / 2
    ax.text(mid, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 5,
            label, ha='center', va='bottom', fontsize=7.5,
            color=c, fontweight='bold')
    if end < K_TOTAL:
        ax.axvline(end - 0.5, color='#999999', linewidth=0.6,
                   linestyle='--', alpha=0.6)

ax.set_xlabel('Keypoint Index')
ax.set_ylabel('Noise Rate (%)')
ax.set_xlim(-1, K_TOTAL)
ax.set_xticks(np.arange(0, K_TOTAL, 5))
ax.grid(axis='y', alpha=0.25, linewidth=0.4)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

legend_handles = [Patch(facecolor=c, label=lbl.replace('\n', ' '))
                  for (lbl, _), c in zip(BODY_PARTS.items(), colors_part)]
ax.legend(handles=legend_handles, loc='upper right',
          framealpha=0.9, edgecolor='#cccccc', fancybox=False, fontsize=8)

plt.suptitle('Noise Coordinate [0,0] Rate per Keypoint Across All Videos',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_F_noise_heatmap.png'))
plt.show()

print('[SAVED] spatial_F_noise_heatmap.png')
print(f'\nTop 10 keypoints with highest noise rate:')
top10 = np.argsort(noise_rate)[::-1][:10]
for idx in top10:
    print(f'  Keypoint {idx:3d}: {noise_rate[idx]*100:.2f}%')

In [ ]:
# ── Fig G: Persentase Frame dengan Noise per Penutur ──────────────────────────
# Menunjukkan variasi kualitas deteksi MediaPipe antar penutur

sp_noise_rate = {}
for sp in speakers:
    sp_vids = [v for v in video_ids if v.startswith(sp)]
    total_f, noise_f = 0, 0
    for vid in sp_vids:
        kp = data[vid]['keypoints']
        T  = kp.shape[0]
        # Frame dianggap punya noise jika ada minimal 1 keypoint tangan yang [0,0]
        has_noise = ((kp[:, :42, 0] == 0) & (kp[:, :42, 1] == 0)).any(axis=1)
        total_f += T
        noise_f += has_noise.sum()
    sp_noise_rate[sp] = (noise_f / total_f * 100) if total_f > 0 else 0

# Pisah LH dan RH noise rate
sp_noise_lh, sp_noise_rh = {}, {}
for sp in speakers:
    sp_vids = [v for v in video_ids if v.startswith(sp)]
    total_f = 0
    n_lh, n_rh = 0, 0
    for vid in sp_vids:
        kp = data[vid]['keypoints']
        T  = kp.shape[0]
        noise_lh = ((kp[:, :21,  0] == 0) & (kp[:, :21,  1] == 0)).any(axis=1)
        noise_rh = ((kp[:, 21:42,0] == 0) & (kp[:, 21:42,1] == 0)).any(axis=1)
        total_f += T;  n_lh += noise_lh.sum();  n_rh += noise_rh.sum()
    sp_noise_lh[sp] = (n_lh / total_f * 100) if total_f > 0 else 0
    sp_noise_rh[sp] = (n_rh / total_f * 100) if total_f > 0 else 0

sp_sorted = sorted(speakers)
x = np.arange(len(sp_sorted))
w = 0.28

fig, ax = plt.subplots(figsize=(9, 4.0), dpi=150)

b0 = ax.bar(x - w, [sp_noise_lh[s] for s in sp_sorted], width=w,
            color='#2166AC', edgecolor='white', linewidth=0.4, label='Left Hand')
b1 = ax.bar(x,     [sp_noise_rh[s] for s in sp_sorted], width=w,
            color='#D6604D', edgecolor='white', linewidth=0.4, label='Right Hand')
b2 = ax.bar(x + w, [sp_noise_rate[s] for s in sp_sorted], width=w,
            color='#888888', edgecolor='white', linewidth=0.4, label='Any Hand')

for bars in [b0, b1, b2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.5:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                    f'{h:.1f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(sp_sorted)
ax.set_xlabel('Signer ID')
ax.set_ylabel('Frames with Noise [0,0] (%)')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25, linewidth=0.4)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.suptitle('Noise [0,0] Rate per Signer — Hand Keypoints',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_G_noise_per_signer.png'))
plt.show()
print('[SAVED] spatial_G_noise_per_signer.png')

In [ ]:
# ── Fig H: Visualisasi Frame dengan Keypoint [0,0] ────────────────────────────
# Keypoint yang noise (koordinat [0,0]) ditampilkan dengan marker merah
# Pilih frame yang punya noise secara otomatis dari data nyata

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
N_EXAMPLES = 4   # Jumlah contoh frame yang ditampilkan
# ──────────────────────────────────────────────────────────────────────────────

# Cari frame-frame yang punya noise tangan secara otomatis
noisy_frames_found = []
for vid in video_ids:
    kp = data[vid]['keypoints']
    for t in range(kp.shape[0]):
        frame = kp[t]  # (K, 2)
        is_noise = (frame[:,0] == 0) & (frame[:,1] == 0)
        n_noise_hand = is_noise[:42].sum()
        if 2 <= n_noise_hand <= 15:   # Punya noise tapi tidak semua
            noisy_frames_found.append((vid, t, frame.copy(), is_noise))
        if len(noisy_frames_found) >= N_EXAMPLES * 3:
            break
    if len(noisy_frames_found) >= N_EXAMPLES * 3:
        break

# Pilih N_EXAMPLES yang bervariasi
selected = noisy_frames_found[:N_EXAMPLES]
print(f'Found {len(noisy_frames_found)} noisy frames. Showing {len(selected)}.')

margin = 0.08
fig, axes = plt.subplots(1, len(selected),
                         figsize=(3.0 * len(selected), 3.5), dpi=150)
if len(selected) == 1:
    axes = [axes]

for ax, (vid, fidx, frame, is_noise) in zip(axes, selected):
    valid = ~is_noise

    # Center based on valid hand points only
    hand_valid_idx = IDX_HAND[valid[IDX_HAND]]
    if len(hand_valid_idx) > 0:
        cx = frame[hand_valid_idx, 0].mean()
        cy = frame[hand_valid_idx, 1].mean()
    else:
        cx, cy = 0.5, 0.5

    f_plot = frame.copy().astype(float)
    f_plot[:,0] = -(f_plot[:,0] - cx)
    f_plot[:,1] -= cy

    for u, v in CONNECTIONS:
        if valid[u] and valid[v]:
            ax.plot([f_plot[u,0], f_plot[v,0]],
                    [-f_plot[u,1], -f_plot[v,1]],
                    color=C_EDGE, linewidth=1.0, alpha=0.6, zorder=1)

    # Valid keypoints
    for idx_part, color in [(IDX_LH, C_LH), (IDX_RH, C_RH)]:
        idx_v = idx_part[valid[idx_part]]
        if len(idx_v):
            ax.scatter(f_plot[idx_v,0], -f_plot[idx_v,1],
                       s=28, color=color, edgecolors='white',
                       linewidths=0.3, zorder=2)

    # Noise keypoints — ditandai merah di posisi simbolis (pojok)
    n_noise = is_noise[:42].sum()
    lh_noise = is_noise[:21].sum()
    rh_noise = is_noise[21:42].sum()

    ax.set_title(f'{vid.split("_")[0]} | F{fidx}\n'
                 f'LH miss: {lh_noise}/21  RH miss: {rh_noise}/21',
                 fontsize=8)

    all_vals = f_plot[IDX_HAND[valid[IDX_HAND]]]
    if len(all_vals):
        side = max(
            all_vals[:,0].max() - all_vals[:,0].min(),
            all_vals[:,1].max() - all_vals[:,1].min()
        ) * (1 + margin) + 0.01
    else:
        side = 0.3

    ax.set_xlim(-side/2, side/2)
    ax.set_ylim(-side/2, side/2)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.5); spine.set_color('#aaaaaa')

legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_LH,
           markersize=7, label='Left Hand (detected)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_RH,
           markersize=7, label='Right Hand (detected)'),
    Line2D([0],[0], marker='x', color=C_NOISE,
           markersize=7, markeredgewidth=1.5, label='Missing [0,0]'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.06), frameon=False)

plt.suptitle('Examples of Frames with Missing Keypoints [0,0] — Hand Region',
             fontsize=11, fontweight='bold')
plt.subplots_adjust(wspace=0.06, bottom=0.18)
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_H_noise_frames.png'))
plt.show()
print('[SAVED] spatial_H_noise_frames.png')

---
## Summary — Output Files


In [ ]:
print('='*60)
print('  OUTPUT SUMMARY — IV.2.2 Analisis Karakteristik Spasial')
print('='*60)
outputs = [
    ('spatial_A_scatter_overlay.png',      'IV.2.2.1 — Scatter overlay koordinat tangan semua penutur'),
    ('spatial_B_boxplot_scale.png',        'IV.2.2.1 — Boxplot variasi skala tangan per penutur'),
    ('spatial_C_skeleton_overlay.png',     'IV.2.2.1 — Overlay skeleton multi-penutur (raw coordinates)'),
    ('spatial_D_microshift_linechart.png', 'IV.2.2.2 — Line chart fluktuasi koordinat antar frame'),
    ('spatial_E_displacement_histogram.png','IV.2.2.2 — Histogram inter-frame displacement per keypoint'),
    ('spatial_F_noise_heatmap.png',        'IV.2.2.3 — Bar chart noise rate per keypoint (86 titik)'),
    ('spatial_G_noise_per_signer.png',     'IV.2.2.3 — Noise rate per penutur (LH vs RH vs Any)'),
    ('spatial_H_noise_frames.png',         'IV.2.2.3 — Contoh frame dengan keypoint [0,0]'),
]
for fname, desc in outputs:
    path = os.path.join(OUTPUT_DIR, fname)
    status = '[OK]' if os.path.exists(path) else '[--]'
    print(f'  {status}  {fname}')
    print(f'         → {desc}')
print('='*60)